# anon_check
k-anonymity, local suppression and l-diversity.

In [21]:
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## Load data

In [22]:
df = pd.read_csv(r"..\data\private_data_preprocessed.csv", dtype=str)   # read everything as text (evote, zip as characters)
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed") or c == "X"], errors="ignore")  # drop row-index column
df["zip"] = df["zip"].str[0] + "***"
df.head()

,name,sex,evote,zip,education,citizenship,marital_status,party,age_group
0,*,Female,0,2***,Medium,Danish,Never married,Green,18-29
1,*,Female,0,2***,Medium,Danish,Never married,Red,45-64
2,*,Female,1,2***,High,Danish,Never married,Green,45-64
3,*,Male,1,2***,High,Danish,Never married,Red,65 or older
4,*,Male,0,2***,Medium,Danish,Never married,Green,30-44


### stuff changed so far  
- removed rows where education not stated
- citizenship binarised
- age binned
- education binned
- name deleted  
- zip left with 2***  
- 


## Quasi-identifiers and sensitive attributes

In [23]:
qi   = ["sex", "evote", "zip", "education", "citizenship", "age_group", "marital_status"]
sens = ["party"]

## k-anonymity check
Size of each equivalence class. `dropna=False` keeps suppressed (`NaN`) values as their own category, like sdcMicro does.

In [24]:
def k_groups(data, qi):
    return data.groupby(qi, dropna=False).size().reset_index(name="n")

K_TARGET = 3

groups = k_groups(df, qi)
print("current k:", groups["n"].min())
print(f"groups violating k = {K_TARGET}:", (groups["n"] < K_TARGET).sum())
print(f"records violating k = {K_TARGET}:", groups.loc[groups["n"] < K_TARGET, "n"].sum())

current k: 1
groups violating k = 3: 79
records violating k = 3: 104


## Local suppression to reach k
sdcMicro's `kAnon()` has no Python equivalent, so this is a simple greedy version: 
for records in classes smaller than k, set one quasi-identifier to `NaN`, starting with the least important column, and repeat until every class has at least k records.

`importance` lists the columns in the order they get suppressed (first = suppressed first). I put the columns with the most categories first, since they cause the most violations. Change it to whatever you can best afford to lose.

In [25]:
def k_anon_suppress(data, qi, k, importance=None):
    out = data.copy()
    order = importance or sorted(qi, key=lambda c: -out[c].nunique())   # most categories first
    for col in order:
        size = out.groupby(qi, dropna=False)[qi[0]].transform("size")
        viol = size < k
        if not viol.any():
            break
        out.loc[viol, col] = pd.NA
    # records still in a class < k: suppress all their quasi-identifiers so they merge into one class
    size = out.groupby(qi, dropna=False)[qi[0]].transform("size")
    out.loc[size < k, qi] = pd.NA
    size = out.groupby(qi, dropna=False)[qi[0]].transform("size")
    if (size < k).any():
        print(f"warning: {(size < k).sum()} fully suppressed records still form a class < k={k}; drop them or lower k")
    return out

df_anon = k_anon_suppress(df, qi, k=K_TARGET)

print("k after suppression:", k_groups(df_anon, qi)["n"].min())
print("NaNs introduced per column:")
print(df_anon[qi].isna().sum())
df_anon.head()

k after suppression: 3
NaNs introduced per column:
sex                 4
evote               4
zip                 0
education          24
citizenship         4
age_group         104
marital_status     50
dtype: int64


,name,sex,evote,zip,education,citizenship,marital_status,party,age_group
0,*,Female,0,2***,Medium,Danish,Never married,Green,18-29
1,*,Female,0,2***,Medium,Danish,Never married,Red,45-64
2,*,Female,1,2***,High,Danish,NaN,Green,NaN
3,*,Male,1,2***,High,Danish,Never married,Red,NaN
4,*,Male,0,2***,NaN,Danish,NaN,Green,NaN


## l-diversity check
Per equivalence class: size and l for the sensitive attribute `party` (number of distinct parties in the class). `marital_status` is a quasi-identifier, so it is part of the class definition and is not checked here.

In [26]:
def l_diversity(data, qi, sens):
    s = sens[0]                                   # only one sensitive attribute: party
    g = data.groupby(qi, dropna=False)
    res = g.agg(n=(s, "size")).reset_index()
    res["l_party"] = g[s].nunique(dropna=False).values
    return res

def l_report(data, qi, sens, l=2):
    lg = l_diversity(data, qi, sens)
    viol = lg["l_party"] < l
    print("current l (party):          ", lg["l_party"].min())
    print("groups violating l (party): ", viol.sum(), "of", len(lg))
    print("records violating l (party):", lg.loc[viol, "n"].sum(), "of", lg["n"].sum())
    return lg

L_TARGET = 2

### On the original data (`df`)

In [27]:
l_groups = l_report(df, qi, sens, L_TARGET)

# the offending groups, smallest first
l_groups[l_groups["l_party"] < L_TARGET].sort_values(["l_party", "n"])
l_groups.head()

current l (party):           1
groups violating l (party):  85 of 103
records violating l (party): 131 of 196


,sex,evote,zip,education,citizenship,age_group,marital_status,n,l_party
0,Female,0,2***,High,Danish,18-29,Never married,3,2
1,Female,0,2***,High,Danish,30-44,Divorced,2,1
2,Female,0,2***,High,Danish,30-44,Married/separated,4,2
3,Female,0,2***,High,Danish,30-44,Never married,2,1
4,Female,0,2***,High,Danish,45-64,Divorced,1,1


### On the k-anonymized data (`df_anon`)

In [28]:
l_groups_anon = l_report(df_anon, qi, sens, L_TARGET)

l_groups_anon[l_groups_anon["l_party"] < L_TARGET].sort_values(["l_party", "n"])

current l (party):           1
groups violating l (party):  19 of 52
records violating l (party): 65 of 196


,sex,evote,zip,education,citizenship,age_group,marital_status,n,l_party
9,Female,0,2***,Medium,Danish,18-29,Never married,3,1
11,Female,0,2***,Medium,Danish,30-44,Never married,3,1
13,Female,0,2***,Medium,Danish,45-64,Never married,3,1
14,Female,0,2***,Medium,Danish,NaN,Divorced,3,1
17,Female,1,2***,High,Danish,30-44,Never married,3,1
24,Male,0,2***,High,Danish,18-29,Never married,3,1
27,Male,0,2***,High,Danish,65 or older,Widowed,3,1
29,Male,0,2***,High,Danish,NaN,NaN,3,1
30,Male,0,2***,High,non-danish,NaN,Married/separated,3,1
31,Male,0,2***,Low,Danish,45-64,Never married,3,1


### Before vs after k-anonymization
Records in classes that fail l-diversity on `party`.

In [29]:
def fail_records(lg, l=L_TARGET):
    return int(lg.loc[lg["l_party"] < l, "n"].sum())

pd.DataFrame({
    "records_total":      [l_groups["n"].sum(), l_groups_anon["n"].sum()],
    "records_failing_l":  [fail_records(l_groups), fail_records(l_groups_anon)],
}, index=["original (df)", "k-anonymized (df_anon)"])

,records_total,records_failing_l
original (df),196,131
k-anonymized (df_anon),196,65


### Skew check
Share of the most common party in each class. A share close to 1 means an attacker can guess the party with high confidence even if l is technically 2.

In [30]:
def max_share(s):
    return s.value_counts(dropna=False).iloc[0] / len(s)

skew = (df_anon.groupby(qi, dropna=False)
        .agg(n=("party", "size"),
             max_share_party=("party", max_share))
        .reset_index())

skew[skew["max_share_party"] > 0.8].sort_values("n").head()

,sex,evote,zip,education,citizenship,age_group,marital_status,n,max_share_party
9,Female,0,2***,Medium,Danish,18-29,Never married,3,1.0
14,Female,0,2***,Medium,Danish,NaN,Divorced,3,1.0
13,Female,0,2***,Medium,Danish,45-64,Never married,3,1.0
11,Female,0,2***,Medium,Danish,30-44,Never married,3,1.0
17,Female,1,2***,High,Danish,30-44,Never married,3,1.0


In [ ]:
df_anon.to_csv(r"..\data\anonymised_dataF.csv", index=False)      # NaN is written as an empty field